In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import sqlite3
import pandas as pd

from src.analytics.cagr import calculate_cagr

DB_PATH = PROJECT_ROOT / "data" / "nifty100.db"

conn = sqlite3.connect(DB_PATH)

profit = pd.read_sql(
    """
    SELECT company_id, year, sales, net_profit, eps
    FROM profitandloss
    """,
    conn
)

profit = profit.sort_values(
    ["company_id", "year"]
)

In [3]:
profit.head()

,company_id,year,sales,net_profit,eps
0,ABB,2012.0,1653.0,145.0,68.0
1,ABB,2014.0,2276.0,198.0,93.0
2,ABB,2015.0,2289.0,229.0,108.0
3,ABB,2016.0,2614.0,255.0,120.0
4,ABB,2017.0,2903.0,277.0,130.0


In [4]:
from src.analytics.cagr import calculate_cagr

results = []

for company, group in profit.groupby("company_id"):

    group = group.sort_values("year").reset_index(drop=True)

    for i in range(len(group)):

        row = group.iloc[i]

        revenue_cagr = None
        revenue_flag = "INSUFFICIENT"

        pat_cagr = None
        pat_flag = "INSUFFICIENT"

        eps_cagr = None
        eps_flag = "INSUFFICIENT"

        if i >= 5:

            revenue_cagr, revenue_flag = calculate_cagr(
                group.iloc[i-5]["sales"],
                row["sales"],
                5
            )

            pat_cagr, pat_flag = calculate_cagr(
                group.iloc[i-5]["net_profit"],
                row["net_profit"],
                5
            )

            eps_cagr, eps_flag = calculate_cagr(
                group.iloc[i-5]["eps"],
                row["eps"],
                5
            )

        results.append({
            "company_id": row["company_id"],
            "year": row["year"],
            "revenue_cagr_5yr": revenue_cagr,
            "revenue_cagr_5yr_flag": revenue_flag,
            "pat_cagr_5yr": pat_cagr,
            "pat_cagr_5yr_flag": pat_flag,
            "eps_cagr_5yr": eps_cagr,
            "eps_cagr_5yr_flag": eps_flag
        })

In [5]:
cagr_df = pd.DataFrame(results)

cagr_df.head(15)

,company_id,year,revenue_cagr_5yr,revenue_cagr_5yr_flag,pat_cagr_5yr,pat_cagr_5yr_flag,eps_cagr_5yr,eps_cagr_5yr_flag
0,ABB,2012.0,NaN,INSUFFICIENT,NaN,INSUFFICIENT,NaN,INSUFFICIENT
1,ABB,2014.0,NaN,INSUFFICIENT,NaN,INSUFFICIENT,NaN,INSUFFICIENT
2,ABB,2015.0,NaN,INSUFFICIENT,NaN,INSUFFICIENT,NaN,INSUFFICIENT
3,ABB,2016.0,NaN,INSUFFICIENT,NaN,INSUFFICIENT,NaN,INSUFFICIENT
4,ABB,2017.0,NaN,INSUFFICIENT,NaN,INSUFFICIENT,NaN,INSUFFICIENT
5,ABB,2018.0,14.814188,NORMAL,22.561840,NORMAL,22.684749,NORMAL
6,ABB,2019.0,10.080782,NORMAL,17.844540,NORMAL,17.915415,NORMAL
7,ABB,2020.0,12.325715,NORMAL,20.960575,NORMAL,20.902725,NORMAL
8,ABB,2021.0,10.518336,NORMAL,22.063993,NORMAL,22.050742,NORMAL
9,ABB,2022.0,11.096390,NORMAL,23.598557,NORMAL,23.665597,NORMAL


In [6]:
cagr_df.shape

(1177, 8)

In [7]:
cagr_df = cagr_df.dropna(subset=["year"])

cagr_df.shape

(1085, 8)

In [9]:
ratio_df = pd.read_sql(
    "SELECT * FROM financial_ratios",
    conn
)

ratio_df.shape

(1044, 16)

In [10]:
ratio_df = ratio_df.merge(
    cagr_df,
    on=["company_id", "year"],
    how="left"
)

ratio_df.shape

(1056, 22)

In [11]:
ratio_df.to_sql(
    "financial_ratios",
    conn,
    if_exists="replace",
    index=False
)

print("financial_ratios updated successfully!")

financial_ratios updated successfully!


In [12]:
pd.read_sql(
    """
    PRAGMA table_info(financial_ratios)
    """,
    conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,company_id,TEXT,0,None,0
1,1,year,REAL,0,None,0
2,2,net_profit_margin_pct,REAL,0,None,0
3,3,operating_profit_margin_pct,REAL,0,None,0
4,4,return_on_equity_pct,REAL,0,None,0
5,5,return_on_assets_pct,REAL,0,None,0
6,6,debt_to_equity,REAL,0,None,0
7,7,interest_coverage,REAL,0,None,0
8,8,asset_turnover,REAL,0,None,0
9,9,free_cash_flow_cr,REAL,0,None,0
